In [2]:
# --- DoE design and variable identification --- #

# --- Imports --- #

import pyDOE3
from pyDOE3 import bbdesign
import numpy as np
import pandas as pd


In [3]:
# --- User Variable settings ---#

response_var = "yield_pct"  #input response variable (Note, using just yield will crash it, dont know why)
results = []  #input the results of response_var following the desgn table (int or float only)

# input relevant factors, use same dictionary format as below (3 or more factors required)
factors = {
    "temp": {"low" : 20, "medium": 50, "high": 75, "units": "C"},
    "cat_mol": {"low": 0.1, "medium": 0.5, "high": 1, "units": "mol"},
    "A_equiv": {"low": 1, "medium": 5, "high": 10, "units": "equiv"}
}



# --- Do not change below --- #
centre_points = 3 # leave as 3
num_factors = len(factors)
factor_names = list(factors.keys())
print(factor_names)
DoE_table = bbdesign(num_factors, center=centre_points)
table = pd.DataFrame(DoE_table, columns=factor_names)
table.insert(0, "StdOrder", range(1, len(table) + 1))

view_table = table.copy()

for f in factor_names:
    view_table.loc[table[f] == -1, f] = factors[f]["low"]
    view_table.loc[table[f] == 0, f] = factors[f]["medium"]
    view_table.loc[table[f] == 1, f] = factors[f]["high"]

print(view_table)
print("experiments may be run following the above table. Once completed, place results in results list")

if len(results) == len(view_table):
    view_table[response_var] = results
    print(view_table)
    print("design table with response variable results attached")
elif 0 < len(results) < len(view_table):
    print("ERROR: incomplete results column, check all results added")


['temp', 'cat_mol', 'A_equiv']
    StdOrder  temp  cat_mol  A_equiv
0          1  20.0      0.1      5.0
1          2  20.0      1.0      5.0
2          3  75.0      0.1      5.0
3          4  75.0      1.0      5.0
4          5  20.0      0.5      1.0
5          6  20.0      0.5     10.0
6          7  75.0      0.5      1.0
7          8  75.0      0.5     10.0
8          9  50.0      0.1      1.0
9         10  50.0      0.1     10.0
10        11  50.0      1.0      1.0
11        12  50.0      1.0     10.0
12        13  50.0      0.5      5.0
13        14  50.0      0.5      5.0
14        15  50.0      0.5      5.0
experiments may be run following the above table. Once completed, place results in results list


In [4]:
# --- Results Profiles --- #

normalised_data_table = view_table.copy()
for f in factor_names:
    midpoint = (factors[f]["low"] + factors[f]["high"]) / 2
    ranges = (factors[f]["high"] - factors[f]["low"])
    normalised_data_table[f] = (2*(view_table[f] - midpoint)) / ranges
#print(normalised_data_table)                                                          
#NB - this method of normalisation seems unique to DoE - google it  2*(x-xmid)/range     

#--- defining the formula for statsmodels ---#
formula_vars = []
formula_interaction_vars = []
formula_quad_vars = []

for f in factor_names:
    formula_vars.append(f)

for i in range(len(factor_names)):
    for j in range(i+1, len(factor_names)):
        formula_interaction_vars.append(f"{factor_names[i]}:{factor_names[j]}")

for f in factor_names:
    formula_quad_vars.append(f"I({f}**2)")

full_formula = formula_vars + formula_interaction_vars + formula_quad_vars
#print(full_formula)
formula = f"{response_var} ~ {' + '.join(full_formula)}"
#print(formula)
        
#print(formula_vars)
#print(formula_interaction_vars)
#print(formula_quad_vars)

# --- ols model ---#
import statsmodels.formula.api as sm
model = sm.ols(formula, data = normalised_data_table).fit()
#model.summary()

#---results plots---#
import numpy as np
import matplotlib.pyplot as plt

grid_points = 150


for i in range(len(factor_names)):
    for j in range(i + 1, len(factor_names)):
        x_factor = factor_names[i]
        y_factor = factor_names[j]

        x_values = np.linspace(factors[x_factor]["low"], factors[x_factor]["high"], grid_points)
        y_values = np.linspace(factors[y_factor]["low"], factors[y_factor]["high"], grid_points)
        X_grid, Y_grid = np.meshgrid(x_values, y_values)

        
        grid_table = pd.DataFrame()
        grid_table[x_factor] = X_grid.ravel()
        grid_table[y_factor] = Y_grid.ravel()
        lab_txt = []
        for f in factor_names:
            if f != x_factor and f != y_factor:
                grid_table[f] = factors[f]["medium"]
                lab_txt.append(f"{f} = {factors[f]["medium"]} {factors[f]["units"]}")

        
        coded_grid = grid_table.copy()
        for f in factor_names:
            midpoint = (factors[f]["low"] + factors[f]["high"]) / 2
            ranges = (factors[f]["high"] - factors[f]["low"])
            coded_grid[f] = (2*(grid_table[f] - midpoint)) / ranges

        
        predicted = model.predict(coded_grid)
        Z_grid = predicted.to_numpy().reshape(X_grid.shape)

        fig = plt.figure(figsize=(12, 5))
        ax1 = fig.add_subplot(1, 2, 1, projection="3d")
        ax1.plot_surface(X_grid, Y_grid, Z_grid, cmap="viridis")
        ax1.set_xlabel(f"{x_factor} ({factors[x_factor]['units']})")
        ax1.set_ylabel(f"{y_factor} ({factors[y_factor]['units']})")
        ax1.set_zlabel(response_var)

        ax2 = fig.add_subplot(1, 2, 2)
        contour = ax2.contourf(X_grid, Y_grid, Z_grid, levels=20, cmap="viridis")
        fig.colorbar(contour, ax=ax2, label=response_var)
        ax2.set_xlabel(f"{x_factor} ({factors[x_factor]['units']})")
        ax2.set_ylabel(f"{y_factor} ({factors[y_factor]['units']})")

        fig.suptitle(f"{x_factor} vs {y_factor}, at: " + ", ".join(lab_txt))
        plt.tight_layout()
        plt.show()
print(model.summary())

PatsyError: Error evaluating factor: NameError: name 'yield_pct' is not defined
    yield_pct ~ temp + cat_mol + A_equiv + temp:cat_mol + temp:A_equiv + cat_mol:A_equiv + I(temp**2) + I(cat_mol**2) + I(A_equiv**2)
    ^^^^^^^^^